# JS13 Praktikum 1 - Artificial Neural Network

Praktikum ini mengimplementasikan jaringan saraf sederhana dua layer untuk mempelajari dataset XOR menggunakan forward pass, backpropagation, dan gradient descent. Tiga konfigurasi dibandingkan: sigmoid dengan 2 neuron hidden (baseline), sigmoid dengan 3 neuron hidden, dan ReLU pada hidden layer dengan 3 neuron.

In [8]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=4, suppress=True)

# Dataset XOR
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([[0], [1], [1], [0]], dtype=float)
X, y

(array([[0., 0.],
        [0., 1.],
        [1., 0.],
        [1., 1.]]),
 array([[0.],
        [1.],
        [1.],
        [0.]]))

## Fungsi aktivasi dan turunannya

Output layer tetap menggunakan sigmoid karena target XOR adalah klasifikasi biner. Parameter `activation` mengatur aktivasi hidden layer sehingga pengaruh ReLU dapat dibandingkan secara terkontrol.

In [9]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))


def sigmoid_derivative(a):
    return a * (1 - a)


def relu(z):
    return np.maximum(0, z)


def relu_derivative(z):
    return (z > 0).astype(float)


In [10]:
def train_xor(hidden_size=2, activation="sigmoid", epochs=10000, lr=0.1, seed=42):
    """Train a two-layer neural network with manual backpropagation."""
    rng = np.random.default_rng(seed)
    W1 = rng.normal(0, 1, (2, hidden_size))
    b1 = np.zeros((1, hidden_size))
    W2 = rng.normal(0, 1, (hidden_size, 1))
    b2 = np.zeros((1, 1))
    losses = []

    for epoch in range(epochs):
        # Forward pass
        z1 = X @ W1 + b1
        if activation == "relu":
            a1 = relu(z1)
        else:
            a1 = sigmoid(z1)
        z2 = a1 @ W2 + b2
        a2 = sigmoid(z2)

        error = y - a2
        loss = np.mean(error ** 2)
        losses.append(loss)

        # Backpropagation
        d_a2 = error * sigmoid_derivative(a2)
        d_W2 = a1.T @ d_a2
        d_b2 = np.sum(d_a2, axis=0, keepdims=True)

        hidden_derivative = (
            relu_derivative(z1)
            if activation == "relu"
            else sigmoid_derivative(a1)
        )
        d_a1 = (d_a2 @ W2.T) * hidden_derivative
        d_W1 = X.T @ d_a1
        d_b1 = np.sum(d_a1, axis=0, keepdims=True)

        # Gradient descent update
        W1 += lr * d_W1
        b1 += lr * d_b1
        W2 += lr * d_W2
        b2 += lr * d_b2

        if epoch % 1000 == 0:
            print(f"Epoch {epoch:5d}, Loss: {loss:.8f}")

    # Forward pass terakhir untuk prediksi
    z1 = X @ W1 + b1
    a1 = relu(z1) if activation == "relu" else sigmoid(z1)
    predictions = sigmoid(a1 @ W2 + b2)
    classes = (predictions >= 0.5).astype(int)

    return {
        "hidden_size": hidden_size,
        "activation": activation,
        "initial_loss": losses[0],
        "final_loss": losses[-1],
        "accuracy": np.mean(classes == y),
        "predictions": predictions,
        "classes": classes,
        "losses": losses,
    }


## Eksperimen 1: baseline sigmoid dengan 2 neuron hidden

In [11]:
baseline = train_xor(hidden_size=2, activation="sigmoid")
print("Prediksi probabilitas:")
print(baseline["predictions"])
print("Kelas:", baseline["classes"].ravel())
print("Akurasi:", f'{baseline["accuracy"]:.2%}')

Epoch     0, Loss: 0.37980460
Epoch  1000, Loss: 0.24847366
Epoch  2000, Loss: 0.23673439
Epoch  3000, Loss: 0.19216475
Epoch  4000, Loss: 0.09713311
Epoch  5000, Loss: 0.03233820
Epoch  6000, Loss: 0.01675629
Epoch  7000, Loss: 0.01085782
Epoch  8000, Loss: 0.00789631
Epoch  9000, Loss: 0.00615042
Prediksi probabilitas:
[[0.0685]
 [0.923 ]
 [0.9249]
 [0.0615]]
Kelas: [0 1 1 0]
Akurasi: 100.00%


## Eksperimen 2: sigmoid dengan 3 neuron hidden

Jumlah neuron hidden diubah dari 2 menjadi 3, sedangkan parameter lain dibuat sama agar perbandingan adil.

In [12]:
sigmoid_3 = train_xor(hidden_size=3, activation="sigmoid")
print("Prediksi probabilitas:")
print(sigmoid_3["predictions"])
print("Kelas:", sigmoid_3["classes"].ravel())
print("Akurasi:", f'{sigmoid_3["accuracy"]:.2%}')

Epoch     0, Loss: 0.24698276
Epoch  1000, Loss: 0.20299021
Epoch  2000, Loss: 0.15385707
Epoch  3000, Loss: 0.07465929
Epoch  4000, Loss: 0.02748283
Epoch  5000, Loss: 0.01392232
Epoch  6000, Loss: 0.00863860
Epoch  7000, Loss: 0.00603794
Epoch  8000, Loss: 0.00455016
Epoch  9000, Loss: 0.00360820
Prediksi probabilitas:
[[0.041 ]
 [0.9293]
 [0.9625]
 [0.0615]]
Kelas: [0 1 1 0]
Akurasi: 100.00%


## Eksperimen 3: ReLU pada hidden layer

Aktivasi hidden layer diganti menjadi ReLU. Sigmoid tetap dipakai pada output layer agar keluaran tetap berupa probabilitas kelas 0 sampai 1.

In [13]:
relu_3 = train_xor(hidden_size=3, activation="relu")
print("Prediksi probabilitas:")
print(relu_3["predictions"])
print("Kelas:", relu_3["classes"].ravel())
print("Akurasi:", f'{relu_3["accuracy"]:.2%}')

Epoch     0, Loss: 0.25140674
Epoch  1000, Loss: 0.00609302
Epoch  2000, Loss: 0.00191740
Epoch  3000, Loss: 0.00108645
Epoch  4000, Loss: 0.00074691
Epoch  5000, Loss: 0.00056486
Epoch  6000, Loss: 0.00045240
Epoch  7000, Loss: 0.00037624
Epoch  8000, Loss: 0.00032139
Epoch  9000, Loss: 0.00028016
Prediksi probabilitas:
[[0.0199]
 [0.99  ]
 [0.99  ]
 [0.0199]]
Kelas: [0 1 1 0]
Akurasi: 100.00%


In [14]:
results = pd.DataFrame([
    {
        "Konfigurasi": "Baseline sigmoid (2 hidden)",
        "Loss awal": baseline["initial_loss"],
        "Loss akhir": baseline["final_loss"],
        "Akurasi": baseline["accuracy"],
    },
    {
        "Konfigurasi": "Sigmoid (3 hidden)",
        "Loss awal": sigmoid_3["initial_loss"],
        "Loss akhir": sigmoid_3["final_loss"],
        "Akurasi": sigmoid_3["accuracy"],
    },
    {
        "Konfigurasi": "ReLU hidden (3 hidden)",
        "Loss awal": relu_3["initial_loss"],
        "Loss akhir": relu_3["final_loss"],
        "Akurasi": relu_3["accuracy"],
    },
])
display(results.assign(**{"Loss awal": results["Loss awal"].round(6), "Loss akhir": results["Loss akhir"].round(6), "Akurasi": (results["Akurasi"] * 100).round(2).astype(str) + "%"}))

,Konfigurasi,Loss awal,Loss akhir,Akurasi
0,Baseline sigmoid (2 hidden),0.379805,0.005012,100.0%
1,Sigmoid (3 hidden),0.246983,0.002968,100.0%
2,ReLU hidden (3 hidden),0.251407,0.000248,100.0%


## Kesimpulan

Penambahan neuron hidden dari 2 menjadi 3 memberi kapasitas representasi yang lebih besar untuk pola XOR. Perbandingan akhir dilihat dari `Loss akhir` dan `Akurasi` pada tabel di atas. ReLU digunakan pada hidden layer dan sigmoid pada output layer; hasilnya dapat dibandingkan langsung dengan konfigurasi sigmoid menggunakan seed, learning rate, dan jumlah epoch yang sama. Karena inisialisasi bobot bersifat acak, penggunaan seed membuat hasil eksperimen ini reproducible.